In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df_train = pd.read_csv('/content/train_cleaned_checkpoint.csv')
df_test = pd.read_csv('/content/test_cleaned_checkpoint.csv')
df_valid = pd.read_csv('/content/valid_cleaned_checkpoint.csv')

# **Label Encoding**


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# fit_transform
y_train = label_encoder.fit_transform(df_train['label'])

# transform
y_valid = label_encoder.transform(df_valid['label'])
y_test  = label_encoder.transform(df_test['label'])

# Class Mapping
class_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
print("Emotion Class Mapping:", class_mapping)

Emotion Class Mapping: {'anger': 0, 'fear': 1, 'happy': 2, 'love': 3, 'sadness': 4}


# **Tokenization & Sequencing (BiLSTM)**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Tokenizer
tokenizer = Tokenizer(oov_token="<OOV>")

# 2. fit
tokenizer.fit_on_texts(df_train['clean_text_dl'])
vocab_size = len(tokenizer.word_index) + 1
print(f"Total Vocabulary Size: {vocab_size}")

# 3. transform
seq_train = tokenizer.texts_to_sequences(df_train['clean_text_dl'])
seq_valid = tokenizer.texts_to_sequences(df_valid['clean_text_dl'])
seq_test  = tokenizer.texts_to_sequences(df_test['clean_text_dl'])


# 4. PADDING & TRUNCATING (43 words)
MAX_LEN = 43

X_train_bilstm = pad_sequences(seq_train, maxlen=MAX_LEN, padding='post', truncating='pre')
X_valid_bilstm = pad_sequences(seq_valid, maxlen=MAX_LEN, padding='post', truncating='pre')
X_test_bilstm  = pad_sequences(seq_test, maxlen=MAX_LEN, padding='post', truncating='pre')

print(f"X_train_bilstm Dimensions: {X_train_bilstm.shape}")

Total Vocabulary Size: 15607
X_train_bilstm Dimensions: (3521, 43)


# **Training Model (BiLSTM + FastText)**

In [ ]:
import tensorflow as tf
from sklearn.metrics import classification_report

## **Class Weight Calculation**

In [ ]:
from sklearn.utils import class_weight

classes = np.unique(y_train)

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights_dict = dict(zip(classes, weights))

print("Weight for each class:")
for class_name, class_id in class_mapping.items():
    print(f"  - Class '{class_name}' (ID: {class_id}) -> Penalty Weight: {class_weights_dict[class_id]:.4f}")

Weight for each class:
  - Class 'anger' (ID: 0) -> Penalty Weight: 0.7993
  - Class 'fear' (ID: 1) -> Penalty Weight: 1.3568
  - Class 'happy' (ID: 2) -> Penalty Weight: 0.8651
  - Class 'love' (ID: 3) -> Penalty Weight: 1.3835
  - Class 'sadness' (ID: 4) -> Penalty Weight: 0.8825


## **FastText Initialization**

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653905 sha256=3586f878f373e9a6f34db5f87de5ce8029a01ee82532172c4e740cbb32e77d0d
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
import urllib.request
import gzip
import os
import fasttext
import fasttext.util
import numpy as np

fasttext.util.download_model('id', if_exists='ignore')

ft_model = fasttext.load_model('cc.id.300.bin')

EMBEDDING_DIM = 300
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

# Mengisi matriks embedding
# get_word_vector() secara otomatis akan mensintesis vektor dari n-gram
# jika kata tersebut adalah Out-Of-Vocabulary (OOV).
for word, i in word_index.items():
    embedding_matrix[i] = ft_model.get_word_vector(word)

print(f"Embedding matrix successfuly build for {vocab_size-1} words.")


Embedding matrix successfuly build for 15606 words.


## **BiLSTM + FastText Architecture**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report

LSTM_UNITS = 64
DROPOUT_RATE = 0.3

model_bilstm_fasttext = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=False # Coba jadi true
    ),
    SpatialDropout1D(DROPOUT_RATE),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=False)),
    Dropout(DROPOUT_RATE),
    Dense(len(class_mapping), activation='softmax')
])

model_bilstm_fasttext.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Inisialisasi Early Stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print("Training BiLSTM + FastText (with Early Stopping)")
history_bilstm_ft = model_bilstm_fasttext.fit(
    X_train_bilstm, y_train,
    validation_data=(X_valid_bilstm, y_valid),
    epochs=30,
    batch_size=32,
    class_weight=class_weights_dict,
    callbacks=[early_stopping],
    verbose=1
)

print("\n[Evaluation BiLSTM + FastText on TEST data]")
pred_probs_ft = model_bilstm_fasttext.predict(X_test_bilstm)
y_pred_bilstm_ft = np.argmax(pred_probs_ft, axis=1)

print(classification_report(y_test, y_pred_bilstm_ft, target_names=label_encoder.classes_))

Training BiLSTM + FastText (with Early Stopping)
Epoch 1/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3366 - loss: 1.4962 - val_accuracy: 0.4750 - val_loss: 1.3064
Epoch 2/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.4893 - loss: 1.2296 - val_accuracy: 0.5295 - val_loss: 1.1748
Epoch 3/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5371 - loss: 1.1379 - val_accuracy: 0.5182 - val_loss: 1.1882
Epoch 4/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5524 - loss: 1.0963 - val_accuracy: 0.5091 - val_loss: 1.1880
Epoch 5/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5652 - loss: 1.0493 - val_accuracy: 0.5523 - val_loss: 1.0415
Epoch 6/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5839 - loss: 1.0193 - val_accuracy: 0.5318 - val_loss: 1.2326
Epoch 7/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5629 - loss: 1.0982 - val_accuracy: 0.5841 - val_loss: 1.0419
Epoch 8/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/

Hasil evaluasi disini berbeda dengan code akhir karena tidak sengaja di-run lagi. Hasil pada NLPFINAL_Evaluation.ipynb merupakan hasil run pertama.

In [ ]:
y_pred_bilstm_ft = np.array(y_pred_bilstm_ft)
np.save('y_pred_bilstm_ft.npy', y_pred_bilstm_ft)